# Reproduce the DACH-CSS historical metrics

No API key, model download or Groq request is needed. This recomputes metrics from stored labels. On Colab, the first cell downloads this public repository after the experiment folder has been uploaded. Read `docs/provenance.md` for the two historical guided-prompt variants.


In [ ]:
from pathlib import Path
import subprocess
import sys

if Path('/content').is_dir():
    repo = Path('/content/roda-viva-evasion')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/ValeriaVSantos/roda-viva-evasion.git', str(repo)], check=True)
    experiment = repo / 'experiments/dach_css_2027'
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    experiment = next((p for p in candidates if (p / 'scripts/evaluate.py').is_file()), None)
    if experiment is None:
        experiment = next((p / 'experiments/dach_css_2027' for p in candidates
                           if (p / 'experiments/dach_css_2027/scripts/evaluate.py').is_file()), None)
if experiment is None or not (experiment / 'scripts/evaluate.py').is_file():
    raise FileNotFoundError('Upload experiments/dach_css_2027 to GitHub first, or open this notebook inside that folder.')
print('Experiment:', experiment)


In [ ]:
output = experiment / 'private_runs/offline_reproduction'
subprocess.run([sys.executable, str(experiment / 'scripts/evaluate.py'), '--out-dir', str(output)], check=True)


In [ ]:
import json
from IPython.display import Markdown, display
display(Markdown((output / 'summary.md').read_text(encoding='utf-8')))
metrics = json.loads((output / 'metrics.json').read_text(encoding='utf-8'))
metrics['corpus']


## Next step

Inspect the confusion matrices in `metrics["subsets"]`. Future inference is a separate, explicit operation through `scripts/run_qwen.py`. Do not execute the archived original notebook as a new-run pipeline.
